# column_tke1: wind driven mixed layer deepening in a single column with the TKE closure
pyOM2: for_config/test_tke1.f90, py_config/test_tke1.py

In [ ]:
import sys; sys.path.append('../..')
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
import pyOM3 as OM
from pyOM3.pe_realistic.model import model
from pyOM3.pe_realistic.tke import tke
from pyOM3.density import linear_eq_of_state_drhodT

In [ ]:
nx, ny, nz = 1, 1, 50
dx, dz, dt = 5e3, 1.0, 360.
N_0        = 0.02
taux       = 1e-4                   # kinematic wind stress
omega, rho_0, grav = 7.2921e-5, 1024., 9.81
nsteps     = 2400                   # 10 days
plot_every = 24

par = dict(nx=nx, ny=ny, nz=nz, dxt=dx, dyt=dx, dzt=dz, dt=dt,
           coord_degree=False, set_zero_at_surface=True,
           periodic_in_x=True, periodic_in_y=True,
           eq_of_state=5, implicit_free_surface=False, explicit_free_surface=False,
           epsilon=1e-12, max_iterations=5000,
           vmix_scheme='implicit', enable_tke=True, alpha_tke=30., mxl_min=1e-8)

In [ ]:
class column_tke1(tke, model):

    def set_coriolis(self):
        f = 2*omega*np.sin(30./180.*np.pi)
        self.coriolis_t = OM.np.asarray(f + 0*self.yt[:, None] + 0*self.xt[None, :])

    def set_initial_conditions(self):
        temp = 32 + rho_0/grav/linear_eq_of_state_drhodT()*(-N_0**2*self.zt_gl)
        temp = temp[:, None, None] + np.zeros((nz, ny, nx))
        zeros = np.zeros((nz, ny, nx))
        super().set_initial_conditions(zeros, zeros, temp, 35. + zeros, np.zeros((ny, nx)))
        self.taux = OM.np.asarray(taux + 0*self.maskU[-self.halo - 1])
        self.tauy = OM.np.zeros_like(self.taux)

    def set_forcing(self, tau):
        ks = -self.halo - 1
        self.du = OM.add_to_array(self.du, (tau, ks), self.taux/self.dzt[ks])

In [ ]:
M = column_tke1(par)
M.set_coriolis()
M.set_topography()
M.prepare_for_run()
M.set_initial_conditions()
M.init_tke()
h = M.halo
rec = []                            # time, temperature and TKE profiles

In [ ]:
def plot(t, tau):
    clear_output(wait=True)
    rec.append((t, M.temp[h:-h, h, h]*1, M.tke[h:-h, h, h]*1))
    fig, ax = plt.subplots(1, 3, figsize=(10, 4))
    ax[0].plot(M.temp[h:-h, h, h], M.zt_gl); ax[0].set_xlabel('T [deg C]'); ax[0].set_ylabel('z [m]')
    ax[1].plot(M.u[h:-h, h, h], M.zt_gl, label='u'); ax[1].plot(M.v[h:-h, h, h], M.zt_gl, label='v')
    ax[1].set_xlabel('[m/s]'); ax[1].legend()
    ax[2].semilogx(M.tke[h:-h, h, h] + 1e-12, M.zw_gl); ax[2].set_xlabel('TKE [m$^2$/s$^2$]')
    fig.suptitle('t = %.1f days' % (t/86400)); plt.tight_layout(); plt.show()

In [ ]:
A, B, C = 1., 0., 0.                 # Adams-Bashforth weights, forward Euler at the start
tau, taum1, taum2 = 2, 1, 0          # pointers to the time levels
for n in range(nsteps):
    M.time_step_implicit_pressure(A, B, C, tau, taum1, taum2)
    if n % plot_every == 0: plot(n*dt, tau)
    tau, taum1, taum2 = (tau+1) % 3, (taum1+1) % 3, (taum2+1) % 3
    A, B, C = (1.5, -0.5, 0.) if n == 0 else (23./12, -16./12, 5./12)

In [ ]:
# temperature as function of time and depth
t = np.array([a[0] for a in rec]); T = np.array([a[1] for a in rec]); e = np.array([a[2] for a in rec])
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
c = ax[0].pcolormesh(t/86400, M.zt_gl, T.T); plt.colorbar(c, ax=ax[0], label='T [deg C]')
c = ax[1].pcolormesh(t/86400, M.zw_gl, np.log10(e.T + 1e-12)); plt.colorbar(c, ax=ax[1], label='log10 TKE')
for a in ax: a.set_xlabel('t [days]'); a.set_ylabel('z [m]')
plt.tight_layout(); plt.show()